# Cross-Dataset Classification PoC

Compare processed scalar metric classifiers across Kaggle, Outfox, and MAGE.
This notebook reads existing metric Parquet files only; it does not run text
processing stages.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from classification_utils import SCALAR_FEATURE_COLUMNS
from logprob_metrics_distributions_utils import load_metrics

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ["TEXT_DETECTION_DATA"])
PROCESSED_ROOT = DATA_ROOT / "processed"

In [ ]:
DATASETS = {
    "kaggle_train": "Kaggle",
    "outfox_chatgpt_train": "Outfox",
    "mage_train": "MAGE",
}
MODELS = ["smollm2_360m", "opt_350m"]
REGIMES = ["Kaggle", "Outfox", "MAGE", "Mixed"]
LABELS = {"HG": 0, "MG": 1}
SEED = 42

In [ ]:
metrics = load_metrics(PROCESSED_ROOT)
metrics = metrics[
    metrics["dataset"].isin(DATASETS)
]
metrics = metrics[metrics["model"].isin(MODELS)]
metrics = metrics[metrics["split"].isin(LABELS)].copy()
metrics["display_dataset"] = metrics["dataset"].map(DATASETS)
coverage = (
    metrics.groupby(["model", "display_dataset", "split"], sort=True)
    .size()
    .rename("rows")
    .reset_index()
)
display(coverage)

In [ ]:
common_features = [
    feature
    for feature in SCALAR_FEATURE_COLUMNS
    if all(feature in metrics.loc[metrics["dataset"] == dataset].columns for dataset in DATASETS)
]
partitions = {}
for model in MODELS:
    for dataset, display_dataset in DATASETS.items():
        cohort = metrics[(metrics["model"] == model) & (metrics["dataset"] == dataset)].copy()
        numeric_features = cohort[common_features].apply(pd.to_numeric, errors="coerce")
        finite_mask = np.isfinite(numeric_features.to_numpy(dtype="float64")).all(axis=1)
        cohort.loc[:, common_features] = numeric_features
        cohort["label"] = cohort["split"].map(LABELS)
        cohort = cohort.loc[finite_mask].reset_index(drop=True)
        if cohort["label"].nunique() != 2:
            continue
        try:
            development, test = train_test_split(
                cohort, test_size=0.20, random_state=SEED, stratify=cohort["label"]
            )
            train, validation = train_test_split(
                development, test_size=0.25, random_state=SEED, stratify=development["label"]
            )
        except ValueError:
            continue
        if any(partition["label"].nunique() != 2 for partition in (train, validation, test)):
            continue
        partitions[(model, display_dataset)] = {
            "train": train.reset_index(drop=True),
            "validation": validation.reset_index(drop=True),
            "test": test.reset_index(drop=True),
        }

In [ ]:
results = []
for model in MODELS:
    available = {
        regime: partitions[(model, regime)]
        for regime in REGIMES[:3]
        if (model, regime) in partitions
    }
    if len(available) >= 2:
        available["Mixed"] = {
            split: pd.concat([partition[split] for partition in available.values()], ignore_index=True)
            for split in ("train", "validation", "test")
        }
    for train_regime, train_partition in available.items():
        classifier = Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ])
        classifier.fit(train_partition["train"][common_features], train_partition["train"]["label"])
        for eval_regime, eval_partition in available.items():
            predictions = classifier.predict(eval_partition["test"][common_features])
            results.append({
                "model": model,
                "train_regime": train_regime,
                "eval_regime": eval_regime,
                "train_rows": len(train_partition["train"]),
                "eval_rows": len(eval_partition["test"]),
                "macro_f1": f1_score(eval_partition["test"]["label"], predictions, average="macro", zero_division=0),
            })
results = pd.DataFrame(results, columns=["model", "train_regime", "eval_regime", "train_rows", "eval_rows", "macro_f1"])
display(results)

In [ ]:
for model in MODELS:
    matrix = (
        results[results["model"] == model]
        .pivot(index="train_regime", columns="eval_regime", values="macro_f1")
        .reindex(index=REGIMES, columns=REGIMES)
    )
    display(matrix)
    sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0.0, vmax=1.0, cmap="viridis", mask=matrix.isna())
    plt.title(f"Cross-dataset macro-F1: {model}")
    plt.xlabel("Evaluation regime")
    plt.ylabel("Training regime")
    plt.show()